# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [8]:





from IPython.display import Markdown, display
from  dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')


from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
from langchain_classic import hub
import os
import tiktoken

load_dotenv()
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4")

# Define safe search function (trim to 3500 tokens)
def safe_search(query: str) -> str:
    """
    Performs a Tavily web search and ensures the result is short enough
    to fit within the GPT-4 token limit.
    """
    result = tavily_search.invoke({"query": query})  # Use invoke instead of run
    
    # Handle result format
    if isinstance(result, list):
        result_text = " ".join([str(item) for item in result])
    elif isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)
    
    # Count and trim tokens
    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])  
    return trimmed



# LangChain tool
tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool")]
#agent = initialize_agent(tools=tools, llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

In [ ]:
# Get the React prompt from hub
prompt = hub.pull("hwchase17/react")

# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create agent executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)





> Entering new AgentExecutor chain...
I need to search the web for the latest research on retrieval-augmented generation in AI.
Action: TavilySafeSearch
Action Input: Latest AI research on retrieval-augmented generation{'title': 'Advancements in RAG: A Comprehensive Survey of ...', 'url': 'https://medium.com/@sahin.samia/advancements-in-rag-a-comprehensive-survey-of-techniques-and-applications-b6160b035199', 'content': 'This report provides an in-depth analysis of the latest and advanced techniques in Retrieval Augmented Generation (RAG), a pivotal approach in enhancing large ...See more', 'score': 0.9993436} {'title': 'Top Research Papers on Retrieval Augmented Generation', 'url': 'https://paperguide.ai/papers/top/research-papers-retrieval-augmented-generation/', 'content': 'Discover the top research papers on Retrieval Augmented Generation. Explore pioneering studies and insights in this dynamic field.', 'score': 0.999015} {'title': 'What Is Retrieval-Augmented Generation aka RAG',

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [11]:
# Your prompt here
prompt_1 = """
You are an expert AI research analyst. Using the TavilySafeSearch tool when needed, retrieve the most recent (2024–2025) developments in generative AI applied to healthcare.

Your task:
1. Perform a web search to gather the latest updates.
2. Analyze the findings.
3. Produce a clear, well-structured summary in **Markdown**.

Your Markdown response must include the following sections:

## Overview
A brief high-level summary of generative AI progress in healthcare.

## Key Advancements in 2025
List and explain the most important breakthroughs or announcements from 2024–2025.

## Clinical Applications
Describe real-world uses in diagnostics, drug discovery, medical imaging, genomics, clinical decision support, etc.

## Notable Research Papers (2024–2025)
Include titles, authors, and a 1–2 sentence summary of each.

## Risks, Challenges, and Limitations
Discuss safety, bias, regulation, reliability, and deployment challenges.

## Sources
List the links or summaries returned by TavilySafeSearch.

Make sure the output is fully formatted in Markdown and suitable for a research report.


"""

# Run the agent
response = agent_executor.invoke({"input": "Latest AI research on retrieval-augmented generation"})
print(response["output"])




> Entering new AgentExecutor chain...
I need to search for the latest research on retrieval-augmented generation in the field of AI.
Action: TavilySafeSearch
Action Input: Latest AI research on retrieval-augmented generation{'title': 'What Is Retrieval-Augmented Generation aka RAG - NVIDIA Blog', 'url': 'https://blogs.nvidia.com/blog/what-is-retrieval-augmented-generation/', 'content': 'The paper, with coauthors from the former Facebook AI Research (now Meta AI), University College London and New York University, called RAG “a general-purpose fine-tuning recipe” because it can be used by nearly any LLM to connect with practically any external resource.\n\n## Building User Trust\n\nRetrieval-augmented generation gives models sources they can cite, like footnotes in a research paper, so users can check any claims. That builds trust. [...] Retrieval-augmented generation is a technique for enhancing the accuracy and reliability of generative AI models with information fetched from specif

In [15]:
from IPython.display import Markdown, display
display(Markdown(response["output"]))


The latest research on retrieval-augmented generation (RAG) in AI involves improving the interface between retrieval and generation components in RAG models. This includes enhancing the models' capacity to selectively source and integrate relevant information from extensive databases. Investigations into more sophisticated retrieval mechanisms, such as bi-directional retrieval and the use of reinforcement learning to optimize query strategies, are underway.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [16]:
# Your prompt here
prompt_2 = """
Use the TavilySafeSearch tool to retrieve the latest (2024–2025) information on emerging AI startups.

Summarize findings in clean Markdown with these sections:

## Top AI Startups (2025)
List each company with:
- Name  
- Focus area  
- Key product or innovation  
- Recent news or milestones  

## Sector Highlights
Briefly note trends across major sectors (e.g., GenAI, healthcare, agentic systems, robotics).

## Funding Signals
Summarize notable funding rounds or investment patterns.

## Sources
Include all sources returned by TavilySafeSearch.

Keep the response concise and clearly formatted.
"""

# Run it
response_2 = agent_executor.invoke({"input": "What are top emerging AI startups and their innovations?"})
print(response["output"])




> Entering new AgentExecutor chain...
I need to search for information about the top emerging AI startups and their innovations.
Action: TavilySafeSearch
Action Input: Top emerging AI startups and their innovations{'title': 'Top AI Startups in 2025 - StartupBlink', 'url': 'https://www.startupblink.com/blog/top-ai-startups/', 'content': '1.   Home\n2.   /\n3.   Innovation Industries\n4.   /\n5.   Artificial Intelligence\n6.   /\n7.   Top AI Startups in…\n\nImage 3: top ai startups\n\n1.   Artificial Intelligence\n2.   Innovation Industries\n\nTop AI Startups in 2025\n\nImage 5: StartupBlink\n\n Written by StartupBlink Posted on June 24, 2025 October 13, 2025Less than 0 min read [...] On our global startup map, more than 11,700 AI startups are represented, accounting for more than 17% of all Software & Data startups, making it the second-largest sub-industry in this category in terms of startup representation. Below, we present the top AI startups of 2025, sorted by their SB Score, whi

In [18]:
display(Markdown(response_2["output"]))


Some of the top emerging AI startups and their innovations include Mistral AI, which specializes in high-performance open-weight large language models; Cursor, which offers AI tools for code writing and debugging; Lovable and Replit, known as "vibe coding" leaders; Thinking Machines Lab, which aims to build more understandable, customizable AI systems; Runway AI Inc., which can turn a text prompt into a high-resolution video; Black Forest Labs, an image-generation startup; and Tennr, an AI automation platform for medical documents.

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [19]:
# Your prompt here
prompt_3 = """
Use the TavilySafeSearch tool to retrieve up-to-date (2024–2025) information comparing OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.

Provide a structured Markdown comparison with:

## Overview
Brief summary of both products.

## Feature Comparison
Side-by-side bullets covering capabilities, models, collaboration tools, limits, and unique features.

## Pricing
Summarize current pricing tiers and what each includes.

## User Feedback & Reviews
Highlight notable strengths, weaknesses, and common user sentiment.

## Recommendation
Short guidance on which product is better suited for different use cases.

## Sources
List all links or snippets returned by TavilySafeSearch.

Keep the response clean, factual, and concise.
"""

# Run it
response_3 = agent_executor.invoke({"input": "Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro"})
print(response["output"])




> Entering new AgentExecutor chain...
I need to find information about the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro. I will use TavilySafeSearch to find this information.
Action: TavilySafeSearch
Action Input: OpenAI’s ChatGPT Team key features, pricing, and reviews{'title': 'ChatGPT Review: Is It Worth It in 2025? [In-Depth]', 'url': 'https://team-gpt.com/blog/chatgpt-review', 'content': 'Team: Starts from $25/month/seat when billed annually, which gives you access to unlimited access to GPT-4o mini and a higher message limit on the other AI models.\n Enterprise: A high-speed access to ChatGPT’s AI models, an expanded context window for longer inputs, enterprise-grade security, and admin controls.\n\n➡️ We dived deeper into ChatGPT’s pricing model to help you figure out if the platform’s pricing structure works for your use case.\n\nRating: 9/10. [...] Integrations: 8/10. ChatGPT’s custom GPTs are a good way of connecting ChatGPT to your

In [20]:
display(Markdown(response_3["output"]))


OpenAI’s ChatGPT Team and Anthropic’s Claude Pro both offer advanced AI capabilities at similar price points. ChatGPT Team starts from $25/month/seat when billed annually and provides unlimited access to GPT-4o mini and a higher message limit on the other AI models. It also offers an enterprise version with high-speed access to ChatGPT’s AI models, an expanded context window for longer inputs, enterprise-grade security, and admin controls. The reviews are generally positive, with some complaints about the free version having limited capabilities and the responses occasionally lacking accuracy when working with complex data.

On the other hand, Anthropic’s Claude Pro is priced at $20 per month or $17 per month when billed annually. It provides enhanced access to Claude’s capabilities, delivering approximately five times the usage of the free tier. The reviews are mixed, with some users praising its capabilities, especially in coding and backend development, while others complain about hitting usage limits within short periods of time. 

In conclusion, both AI models have their strengths and weaknesses, and the choice between the two would depend on the specific needs and requirements of the user.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
